# 01 Exploratory Data Analysis (EDA)
## Nifty Financial Platform

This notebook provides a comprehensive overview of the financial data stored in our PostgreSQL warehouse. We will explore revenue distributions, profit margins, and sector-wise performance.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Settings
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load environment variables
load_dotenv(dotenv_path='../.env')
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    print("ERROR: DATABASE_URL not found in .env file")
else:
    engine = create_engine(DATABASE_URL)
    print("Database engine created successfully.")

## 1. Load Warehouse Data
We will pull data from the main fact and dimension tables.

In [ ]:
def load_master_data(engine):
    query = """
    SELECT 
        dc.symbol, 
        dc.company_name, 
        ds.sector_name, 
        dy.fiscal_year,
        pl.revenue, 
        pl.net_profit, 
        pl.net_profit_margin_pct,
        bs.total_assets, 
        bs.debt_to_equity,
        an.market_cap,
        an.stock_pe
    FROM fact_profit_loss pl
    JOIN dim_company dc ON pl.company_id = dc.company_id
    JOIN dim_sector ds ON dc.sector_id = ds.sector_id
    JOIN dim_year dy ON pl.year_id = dy.year_id
    JOIN fact_balance_sheet bs ON pl.company_id = bs.company_id AND pl.year_id = bs.year_id
    JOIN fact_analysis an ON pl.company_id = an.company_id AND pl.year_id = an.year_id
    """
    return pd.read_sql(query, engine)

df = load_master_data(engine)
print(f"Loaded {len(df)} records.")
df.head()

## 2. Basic Statistics

In [ ]:
print("--- Data Summary ---")
display(df.describe())

print("--- Missing Values ---")
print(df.isnull().sum())

## 3. Revenue & Profit Analysis

In [ ]:
# Top 10 companies by Revenue
top_revenue = df[df['fiscal_year'] == df['fiscal_year'].max()].sort_values('revenue', ascending=False).head(10)
sns.barplot(data=top_revenue, x='revenue', y='company_name', palette='viridis')
plt.title(f'Top 10 Companies by Revenue ({df["fiscal_year"].max()})')
plt.show()

In [ ]:
# Sector-wise Average Profit Margin
plt.figure(figsize=(12, 8))
sector_margin = df.groupby('sector_name')['net_profit_margin_pct'].mean().sort_values()
sector_margin.plot(kind='barh', color='skyblue')
plt.title('Average Net Profit Margin % by Sector')
plt.xlabel('Margin %')
plt.show()

## 4. Correlation Analysis
Let's see how different financial metrics relate to each other.

In [ ]:
corr = df[['revenue', 'net_profit', 'total_assets', 'market_cap', 'debt_to_equity']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Key Financial Metrics')
plt.show()

## 5. Market Cap vs Profitability
Interactive visualization using Plotly.

In [ ]:
fig = px.scatter(df[df['fiscal_year'] == df['fiscal_year'].max()], 
                 x='market_cap', y='net_profit', 
                 size='total_assets', color='sector_name', 
                 hover_name='company_name', log_x=True, log_y=True,
                 title='Market Cap vs Net Profit (Bubble size = Total Assets)')
fig.show()

## 6. Export Summary Data
Saving the aggregated sector performance to a CSV for quick reference.

In [ ]:
sector_summary = df.groupby('sector_name').agg({
    'revenue': 'sum',
    'net_profit': 'sum',
    'market_cap': 'sum',
    'net_profit_margin_pct': 'mean'
}).reset_index()

os.makedirs('../data/analysis', exist_ok=True)
sector_summary.to_csv('../data/analysis/sector_eda_summary.csv', index=False)
print("EDA summary exported to data/analysis/sector_eda_summary.csv")